# Personal AI Assistant

## A Comprehensive Knowledge Worker combining RAG and Function Calling

This Personal AI assistant combines two powerful approaches:
1. **RAG (Retrieval Augmented Generation)**: Uses a vector database to retrieve relevant information from your knowledge base
2. **Function Calling**: Enables the AI to record user details and track unknown questions

The assistant acts as your personal representative, answering questions about your background, experience, and skills while engaging professionally with potential clients or employers.

In [1]:
# Import all required libraries

import os
import glob
import json
import requests
from dotenv import load_dotenv
import gradio as gr
from pypdf import PdfReader
import numpy as np

# LangChain imports for RAG
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# OpenAI imports for function calling
from openai import OpenAI

In [ ]:
# Configuration

MODEL = "gpt-5-mini"  # Cost-effective model for production
DB_NAME = "vector_db"
KNOWLEDGE_BASE_PATH = "Knowledge_Base"
PDF_PATH = "me/linkedin.pdf"
SUMMARY_PATH = "me/summary.txt"

# Performance settings
CHUNK_SIZE = 800  # Smaller chunks for better precision (was 1000)
CHUNK_OVERLAP = 150  # Reduced overlap for less redundancy
TOP_K_RETRIEVAL = 3  # Retrieve top 3 most relevant chunks (was 4)
MAX_CONTEXT_LENGTH = 3000  # Limit total context length

In [3]:
# Load environment variables

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

# Initialize OpenAI client
openai = OpenAI()

# Initialize Pushover for notifications (optional)
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user and pushover_token:
    print(f"✓ Pushover notifications enabled")
else:
    print("ℹ Pushover notifications not configured (optional)")

✓ Pushover notifications enabled


## Step 1: Build the RAG Knowledge Base

Load documents from the Knowledge_Base folder and create a vector store for retrieval.

In [4]:
# Load documents from Knowledge_Base folder

folders = glob.glob(f"{KNOWLEDGE_BASE_PATH}/*")

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

# Handle encoding for different systems
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work on Windows, try: text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

# Split documents into chunks with optimized settings
text_splitter = CharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(documents)

print(f"✓ Loaded {len(documents)} documents")
print(f"✓ Split into {len(chunks)} chunks")
print(f"✓ Document types: {set(doc.metadata['doc_type'] for doc in documents)}")

Created a chunk of size 834, which is longer than the specified 800
Created a chunk of size 980, which is longer than the specified 800
Created a chunk of size 876, which is longer than the specified 800
Created a chunk of size 980, which is longer than the specified 800
Created a chunk of size 876, which is longer than the specified 800
Created a chunk of size 10824, which is longer than the specified 800
Created a chunk of size 3250, which is longer than the specified 800
Created a chunk of size 922, which is longer than the specified 800
Created a chunk of size 1224, which is longer than the specified 800
Created a chunk of size 2852, which is longer than the specified 800
Created a chunk of size 1112, which is longer than the specified 800
Created a chunk of size 1013, which is longer than the specified 800
Created a chunk of size 5065, which is longer than the specified 800
Created a chunk of size 971, which is longer than the specified 800
Created a chunk of size 1106, which is l

✓ Loaded 46 documents
✓ Split into 520 chunks
✓ Document types: {'Education', 'Extra', 'Projects', 'Narrative & Storytelling', 'Skills', 'Career_Goals', 'Publication_Research', 'Work Experience', '_reference', 'Background', 'Achievements & Recognition', 'Interests & Hobbies'}


In [5]:
# Create vector store with embeddings

embeddings = OpenAIEmbeddings()

# Alternative: Use free HuggingFace embeddings
# from langchain_community.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Delete existing database if it exists
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"✓ Cleared existing vector store")

# Create new vectorstore
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_NAME)
print(f"✓ Vector store created with {vectorstore._collection.count()} documents")

# Create retriever for RAG with optimized settings
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RETRIEVAL}
)

✓ Cleared existing vector store
✓ Vector store created with 520 documents
✓ Vector store created with 520 documents


## Step 2: Visualize the Vector Store (Optional)

Visualize how the documents are organized in the vector space.

In [6]:
# Prepare data for visualization

collection = vectorstore._collection
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents_text = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]

# Define color mapping for different document types
color_map = {
    "_reference": "grey",
    "Achievements & Recognition": "gold",
    "Background": "blue",
    "Career_Goals": "teal",
    "Education": "purple",
    "Extra": "navy",
    "Interests & Hobbies": "green",
    "Narrative & Storytelling": "orange",
    "Projects": "red",
    "Publication_Research": "brown",
    "Skills": "cyan",
    "Work Experience": "magenta",
}
colors = [color_map.get(t, "black") for t in doc_types]

sample_embedding = result['embeddings'][0]
print(f"✓ Vector store contains {len(vectors):,} vectors with {len(sample_embedding):,} dimensions each")

✓ Vector store contains 520 vectors with 1,536 dimensions each


In [7]:
# Create 3D visualization using t-SNE

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents_text)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Vector Store Visualization - Knowledge Base',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Step 3: Load LinkedIn Profile and Summary

Extract information from PDF and text files.

In [8]:
# Extract LinkedIn profile from PDF

reader = PdfReader(PDF_PATH)
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

# Read summary text file
with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = f.read()

# Extract name (can be configured)
name = "Hoang Nhat Duy Le"

print(f"✓ Loaded LinkedIn profile ({len(linkedin)} characters)")
print(f"✓ Loaded summary ({len(summary)} characters)")
print(f"✓ Personal AI configured for: {name}")

✓ Loaded LinkedIn profile (5202 characters)
✓ Loaded summary (345 characters)
✓ Personal AI configured for: Hoang Nhat Duy Le


## Step 4: Define Function Calling Tools

Set up functions for recording user details and unknown questions.

In [9]:
# Notification function

def push(message):
    """Send push notification via Pushover"""
    print(f"📱 Push: {message}")
    if pushover_user and pushover_token:
        payload = {"user": pushover_user, "token": pushover_token, "message": message}
        try:
            requests.post(pushover_url, data=payload)
        except Exception as e:
            print(f"Warning: Could not send push notification: {e}")

In [10]:
# Tool functions

def record_user_details(email, name="Name not provided", notes="not provided"):
    """Record user contact information and interest"""
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok", "message": "Thank you! I've recorded your details and will be in touch."}

def record_unknown_question(question):
    """Record questions that couldn't be answered"""
    push(f"Recording unknown question: {question}")
    return {"recorded": "ok", "message": "I've noted this question for follow-up."}

In [11]:
# Define tool schemas for OpenAI function calling

record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [12]:
# Function to handle tool calls

def handle_tool_calls(tool_calls):
    """Execute tool calls and return results"""
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"🔧 Tool called: {tool_name}")
        
        # Dynamically get the function from globals
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {"error": "Tool not found"}
        
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

## Step 5: Build the Combined System Prompt

Create a comprehensive system prompt that incorporates both RAG context and function calling.

In [13]:
# Create a concise system prompt (LinkedIn removed to reduce token usage)

system_prompt = f"""You are acting as {name}'s personal AI assistant. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills, and experience.

Your responsibilities:
1. Represent {name} faithfully and professionally for all interactions
2. Answer questions using ONLY the provided context from the knowledge base
3. Be concise and engaging, as if talking to a potential client or future employer
4. If you don't know the answer, use your record_unknown_question tool
5. If the user wants to connect, ask for their email and use the record_user_details tool

## Brief Summary:
{summary}

Note: Detailed information is provided in the retrieved context below.
"""

print(f"✓ System prompt created ({len(system_prompt)} characters)")

✓ System prompt created (1082 characters)


## Step 6: Create the Combined Chat Function

Integrate RAG retrieval with OpenAI function calling.

In [14]:
# Main chat function with streaming and optimizations

def chat(message, history):
    """
    Optimized chat function with streaming that:
    1. Retrieves relevant context from the vector store (RAG)
    2. Streams responses from OpenAI with function calling
    3. Handles tool calls as needed
    """
    
    # Step 1: Retrieve relevant documents using RAG
    docs = retriever.invoke(message)
    
    # Optimize context: only include most relevant parts, limit total length
    context_parts = []
    total_length = 0
    for doc in docs:
        doc_content = doc.page_content
        doc_type = doc.metadata.get('doc_type', 'Unknown')
        context_part = f"[{doc_type}]: {doc_content}"
        
        if total_length + len(context_part) > MAX_CONTEXT_LENGTH:
            # Truncate to fit
            remaining = MAX_CONTEXT_LENGTH - total_length
            if remaining > 100:  # Only add if meaningful content fits
                context_part = context_part[:remaining] + "..."
                context_parts.append(context_part)
            break
        
        context_parts.append(context_part)
        total_length += len(context_part)
    
    context = "\n\n".join(context_parts)
    
    # Step 2: Build conversation history (limit to last 3 exchanges)
    history_text = ""
    if history:
        recent_history = history[-6:]  # Last 3 exchanges (6 messages)
        for msg in recent_history:
            role = msg.get("role", "")
            content = msg.get("content", "")
            if role == "user":
                history_text += f"User: {content}\n"
            elif role == "assistant":
                history_text += f"Assistant: {content}\n"
    
    # Step 3: Create concise user message with retrieved context
    user_message_with_context = f"""Context: {context}

Question: {message}"""
    
    if history_text:
        user_message_with_context = f"Recent conversation:\n{history_text}\n\n" + user_message_with_context
    
    # Step 4: Build messages array
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message_with_context}
    ]
    
    # Step 5: Stream OpenAI response with function calling
    done = False
    max_iterations = 5
    iteration = 0
    final_response = ""
    
    while not done and iteration < max_iterations:
        iteration += 1
        
        # Use streaming for the final response
        stream = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            temperature=0.7,
            stream=True  # Enable streaming
        )
        
        # Collect streamed response
        collected_messages = []
        tool_calls_data = []
        finish_reason = None
        
        for chunk in stream:
            if chunk.choices:
                delta = chunk.choices[0].delta
                finish_reason = chunk.choices[0].finish_reason
                
                # Handle content
                if delta.content:
                    collected_messages.append(delta.content)
                    final_response = "".join(collected_messages)
                    yield final_response  # Stream to UI - yield the current accumulated message
                
                # Handle tool calls
                if delta.tool_calls:
                    for tool_call_chunk in delta.tool_calls:
                        # Initialize or update tool call data
                        if len(tool_calls_data) <= tool_call_chunk.index:
                            tool_calls_data.append({
                                "id": "",
                                "function": {"name": "", "arguments": ""}
                            })
                        
                        tc = tool_calls_data[tool_call_chunk.index]
                        if tool_call_chunk.id:
                            tc["id"] = tool_call_chunk.id
                        if tool_call_chunk.function:
                            if tool_call_chunk.function.name:
                                tc["function"]["name"] = tool_call_chunk.function.name
                            if tool_call_chunk.function.arguments:
                                tc["function"]["arguments"] += tool_call_chunk.function.arguments
        
        # Check if we need to handle tool calls
        if finish_reason == "tool_calls" and tool_calls_data:
            # Create tool call objects
            from types import SimpleNamespace
            tool_calls = []
            for tc_data in tool_calls_data:
                tool_call = SimpleNamespace(
                    id=tc_data["id"],
                    function=SimpleNamespace(
                        name=tc_data["function"]["name"],
                        arguments=tc_data["function"]["arguments"]
                    )
                )
                tool_calls.append(tool_call)
            
            # Handle the tool calls
            results = handle_tool_calls(tool_calls)
            
            # Create message object for tool calls
            message_obj = {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    }
                    for tc in tool_calls
                ]
            }
            
            messages.append(message_obj)
            messages.extend(results)
            # Reset for next iteration
            collected_messages = []
            final_response = ""
        else:
            done = True
    
    # Return final response (this ensures history is preserved)
    if final_response:
        return final_response
    else:
        return "I apologize, but I couldn't generate a response. Please try again."

## Step 7: Launch the Gradio Interface

Create an interactive web interface for chatting with the Personal AI.

In [15]:
# Optional: Force dark mode
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

In [16]:
# Launch the Gradio chat interface

print("🚀 Launching Personal AI Assistant...")
print(f"📚 Knowledge base: {len(chunks)} chunks from {len(documents)} documents")
print(f"🤖 Model: {MODEL}")
print(f"👤 Acting as: {name}")

view = gr.ChatInterface(
    chat, 
    type="messages",
    title=f"Personal AI - {name}",
    description="Ask me anything about my background, experience, skills, and projects!",
    js=force_dark_mode
).launch(inbrowser=True, share=True)

🚀 Launching Personal AI Assistant...
📚 Knowledge base: 520 chunks from 46 documents
🤖 Model: gpt-4o-mini
👤 Acting as: Hoang Nhat Duy Le
* Running on local URL:  http://127.0.0.1:7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://33ce6055e1bbab1e8d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
* Running on public URL: https://33ce6055e1bbab1e8d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
